# Understanding SAE with your VoiceMOS emotional TTS recordings

**Goal:** follow one real recording from waveform → HuBERT hidden activations → sparse SAE features → reconstructed hidden activations. Then investigate what activates a feature across your recordings.

This is an educational, exploratory analysis of a small, emotion-balanced **training-set subset**. We load a pretrained HuBERT and a pretrained AudioSAE; we do not train or modify TransportMOS, infer quality from reconstruction, or claim a feature has a verified semantic meaning.

The selected model is HuBERT-base, **not your TransportMOS WavLM/M2D2 encoder or MOS head**. We are learning the analysis workflow before applying it to your actual evaluator. An SAE must match the backbone and extraction location it was trained on.

Adapted from [AudioSAE interpretability notebook](https://github.com/audiosae/audio-sae/blob/main/examples/interpretability.ipynb). Vendored helpers were retrieved from that repository on 2026-11-22. Upstream HEAD observed during setup: `5c6708f487244a581397e612e4fce73e70f127e0` (the helper-file hashes below identify the exact copies used).

**How to use:** run top to bottom. Read each “How to read” note after its plot. Re-running uses the saved activation cache for the same selection and checkpoint.


## 1. Setup: the two meanings of activation

A **HuBERT activation** is a numerical coordinate of its internal representation at an audio time frame. One coordinate may mix multiple factors.

An **SAE activation** is the response strength of a learned dictionary feature at that frame. It is not a probability, MOS, or named acoustic measurement. Feature numbers are addresses within this particular checkpoint, not universal concepts.

Use the existing research environment: `your Python environment`. The execution record below reports the actual software and device.


In [ ]:
import os, sys, json, csv, math, hashlib, platform
from pathlib import Path
from collections import Counter, defaultdict
BASE = Path(os.environ.get("SPEECHSAE_BASE", str(Path.cwd())))
BASE.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE / 'vendor'))
os.environ.setdefault('HF_HOME', str(BASE / 'hf_cache'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import numpy as np
import torch
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import resample_poly
from IPython.display import display, Audio, Markdown, Image
from transformers import HubertModel
from huggingface_hub import hf_hub_download
from audio_sae import BatchTopKSAE, reconstruction_metrics, feature_activation_windows
from audio_sae.models import MyHubert
SEED=20261122
np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DATA = Path(os.environ.get("SPEECHSAE_DATA", str(BASE / "Data/vmc2026-track2")))
FIGS = BASE / 'figures'; FIGS.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':110, 'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})
def show(fig, name):
    fig.savefig(FIGS / (name+'.png'), bbox_inches='tight')
    display(Image(filename=str(FIGS / (name+'.png')))); plt.close(fig)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
print('Python:',sys.executable)
print('Torch:',torch.__version__, '| Device:', DEVICE)
print('Data exists:', DATA.is_dir())
print(torch.cuda.is_available())
assert DATA.is_dir(), DATA


## 2. Select your recordings and keep their labels

We use up to **8 recordings per emotion** from `sets/train.csv`, randomly sampled with a fixed seed. Each clip is resampled to 16 kHz and limited to its first 10 seconds to make the walkthrough practical. The manifest records the exact filenames, intended emotion, text, system, speaker, and available listener means.

`emotion` is the intended emotion in metadata; listeners may perceive something different. MOS labels are displayed as context only and do not train the pretrained SAE. Only recordings with training judgments are eligible.


In [ ]:
SR=16000; MAX_SECONDS=10; PER_EMOTION=8
metadata={}
with open(DATA/'metadata.csv',newline='') as f:
    for row in csv.reader(f,delimiter='|'):
        if len(row)>=3 and row[0].endswith('.wav'):
            metadata[row[0]]={'wavID':row[0], 'emotion':row[1], 'text':'|'.join(row[2:])}
rating_rows=defaultdict(list)
with open(DATA/'sets/train.csv',newline='') as f:
    for row in csv.DictReader(f,delimiter='|'): rating_rows[row['wavID']].append(row)
groups=defaultdict(list)
for name,m in metadata.items():
    if name in rating_rows and (DATA/'wav'/name).is_file(): groups[m['emotion']].append(name)
rng=np.random.default_rng(SEED)
records=[]
for emotion,names in sorted(groups.items()):
    chosen=rng.choice(sorted(names),size=min(PER_EMOTION,len(names)),replace=False)
    for name in chosen:
        r=dict(metadata[name]); parts=name.split('-')
        r.update(path=str(DATA/'wav'/name),system=parts[2],speaker=parts[3])
        for key in ['qMOS','eMOS']:
            values=[float(v[key]) for v in rating_rows[name] if v.get(key,'')]
            r[key]=float(np.mean(values)) if values else None
        records.append(r)
assert len(records)>=6, 'Not enough matching training recordings'
(BASE/'selected_recordings.json').write_text(json.dumps(records,indent=2))
print('Available training recordings by intended emotion:', {k:len(v) for k,v in groups.items()})
print('Selected:',len(records),dict(Counter(r['emotion'] for r in records)))
fig,ax=plt.subplots(figsize=(9,3))
counts=Counter(r['emotion'] for r in records)
ax.bar(list(counts),list(counts.values()),color='#3973ac'); ax.set_ylabel('Recordings'); ax.set_title('Your exploratory subset: intended emotions')
ax.tick_params(axis='x',rotation=25); fig.tight_layout();show(fig,'01_dataset_emotions')
def read_audio(path):
    wav,sr=sf.read(path,dtype='float32',always_2d=True); wav=wav.mean(axis=1)
    if sr!=SR:
        g=math.gcd(sr,SR);wav=resample_poly(wav,SR//g,sr//g)
    return np.asarray(wav[:MAX_SECONDS*SR],dtype=np.float32)
example_index=next((i for i,r in enumerate(records) if r['emotion'].lower()=='happy'),0)
example=records[example_index]; waveform=read_audio(example['path'])
print(json.dumps(example,indent=2)); print('Analyzed duration:',len(waveform)/SR,'seconds')
display(Audio(waveform,rate=SR,normalize=False))


## 3. Listen and look at your audio

The waveform shows amplitude over time. The log-mel spectrogram summarizes energy across frequency bands over time. These are descriptions of the **audio itself**, not neural activations. Later we align a feature's response to this same timeline.


In [ ]:
import librosa
mel=librosa.feature.melspectrogram(y=waveform,sr=SR,n_mels=80,hop_length=160)
logmel=librosa.power_to_db(mel,ref=np.max)
duration=len(waveform)/SR
fig,axes=plt.subplots(2,1,figsize=(11,5),sharex=True)
axes[0].plot(np.arange(len(waveform))/SR,waveform,lw=.6,color='#3973ac'); axes[0].set_ylabel('Amplitude')
axes[0].set_title(f"{example['emotion']} | {example['wavID']}")
im=axes[1].imshow(logmel,origin='lower',aspect='auto',extent=[0,duration,0,80],cmap='magma',vmin=-80,vmax=0)
axes[1].set(xlabel='Time (seconds)',ylabel='Mel band')
fig.colorbar(im,ax=axes[1],label='dB relative to clip peak');show(fig,'02_waveform_spectrogram')


## 4. Extract hidden activations and encode them with AudioSAE

The released demonstration uses checkpoint `layer_11/ae.pt`. We use the exact upstream `MyHubert` wrapper convention: `sae_after_layer=11` selects Hugging Face hidden-state tuple index `10` in this code. This is recorded explicitly because layer numbering conventions differ.

For each time frame, HuBERT-base returns 768 coordinates. The SAE expands these to 6,144 features and keeps a sparse subset using the checkpoint's stored inference threshold. Thus `k=50` is a training sparsity target; the observed active count can vary by frame.

The decoder maps sparse features back to 768 coordinates. It reconstructs **HuBERT's hidden state**, not the waveform. This checkpoint does not produce MOS predictions.


In [ ]:
LAYER=11
CKPT=BASE/'ae_layer11.pt'
if not CKPT.is_file():
    CKPT=Path(hf_hub_download(repo_id='Egorgij21/Audio-SAE-hubert-base', filename=f'layer_{LAYER}/ae.pt'))
hubert=MyHubert('facebook/hubert-base-ls960',sae_after_layer=LAYER).to(DEVICE).eval()
sae=BatchTopKSAE.from_pretrained(str(CKPT),device=DEVICE).eval()
for model in (hubert,sae):
    for p in model.parameters():p.requires_grad_(False)
print('Checkpoint SHA256:',sha(CKPT))
print('Hidden-state tuple index:',hubert.sae_after_layer)
print('Input dimension:',sae.activation_dim,'| Features:',sae.dict_size,'| k:',int(sae.k.item()),'| Threshold:',float(sae.threshold.item()))
assert sae.activation_dim==768
with torch.inference_mode():
    acts=hubert(torch.from_numpy(waveform)[None].to(DEVICE))
    feats=sae.encode(acts,use_threshold=True)
    recon=sae.decode(feats)
print('acts:',tuple(acts.shape),'feats:',tuple(feats.shape),'recon:',tuple(recon.shape))
metrics=reconstruction_metrics(acts,feats,recon)
print('Reconstruction metrics:',metrics)
A=acts[0].cpu().numpy(); F=feats[0].cpu().numpy(); R=recon[0].cpu().numpy()
frame_times=np.arange(len(A))/50.0


In [ ]:
acts

### How one real time frame becomes a sparse code

The two strips below show **all 768 original coordinates** and **all 6,144 SAE feature activations** at one frame. They have different dimensions and different color scales. Most sparse cells should be zero. The lower bar plot magnifies only the ten strongest SAE responses at that same frame.

A feature's large value means a strong response in this checkpoint; it does not tell us its meaning. No semantic labels have been assigned here.


In [ ]:
frame=int(np.argmax(F.max(axis=1)))
fig,axes=plt.subplots(3,1,figsize=(12,6),gridspec_kw={'height_ratios':[1,1,2]})
scale=max(abs(A[frame]).max(),1e-6)
im0=axes[0].imshow(A[frame:frame+1],aspect='auto',cmap='RdBu_r',vmin=-scale,vmax=scale)
axes[0].set(yticks=[],xlabel='HuBERT coordinate (768 total)',title=f'One frame at approximately {frame/50:.2f} seconds: dense hidden activation')
fig.colorbar(im0,ax=axes[0],label='Signed value',fraction=.03,pad=.02)
im1=axes[1].imshow(F[frame:frame+1],aspect='auto',cmap='magma',vmin=0)
axes[1].set(yticks=[],xlabel='SAE feature ID (6,144 total)',title=f'Sparse feature activations: {np.count_nonzero(F[frame])} nonzero')
fig.colorbar(im1,ax=axes[1],label='Response',fraction=.03,pad=.02)
top=np.argsort(F[frame])[-10:][::-1]
axes[2].bar([f'F{i}' for i in top],F[frame,top],color='#3973ac');axes[2].set(ylabel='Activation',title='Largest feature responses at this same frame')
fig.tight_layout();show(fig,'03_dense_to_sparse')


### Zoom: dense HuBERT coordinates 100–200

This is the **same recording and same frame** as the previous plot. Both endpoints are included (101 coordinates, zero-based indexing). The horizontal axis lists coordinates, not time. Red bars are positive, blue bars negative; bar height shows the numerical activation. These coordinates have no assigned semantic labels. The strip retains the full-vector color scale for comparison.


In [ ]:
# Same frame as the dense-to-sparse plot; zero-based coordinates, inclusive endpoints.
start_coordinate, end_coordinate = 100, 200
coordinates = np.arange(start_coordinate, end_coordinate + 1)
values = A[frame, coordinates]
fig, axes = plt.subplots(2, 1, figsize=(12, 5), gridspec_kw={'height_ratios': [1, 3]})
scale = max(float(np.abs(A[frame]).max()), 1e-6)
im = axes[0].imshow(values[None, :], aspect='auto', cmap='RdBu_r', vmin=-scale, vmax=scale,
                    extent=[99.5, 200.5, 0, 1], interpolation='nearest')
axes[0].set(yticks=[], title=f'HuBERT dense activation: coordinates 100–200 at {frame/50:.2f} seconds')
fig.colorbar(im, ax=axes[0], label='Signed activation', fraction=.025, pad=.02)
colors = np.where(values >= 0, '#c23c3c', '#3973ac')
axes[1].bar(coordinates, values, color=colors, width=.8)
axes[1].axhline(0, color='gray', lw=.8)
axes[1].set(xlim=(99,201), xlabel='HuBERT coordinate (zero-based index)', ylabel='Activation value')
axes[1].set_xticks(np.arange(100,201,10))
fig.tight_layout()
show(fig, '09_dense_coordinates_100_200')
print(f'{len(coordinates)} coordinates, minimum {values.min():.3f}, maximum {values.max():.3f}.')


## 5. Can the sparse code reconstruct the original hidden state?

In the scatter plot, each dot is one HuBERT coordinate at the selected frame. Points on the diagonal would indicate perfect reconstruction. The time plot shows relative reconstruction error at each frame.

**L0** counts active SAE features. **L2** measures vector reconstruction error. **Normalized L2** divides by input-vector norm. **Fraction of variance explained** compares residual variance with original activation variance. These are not speech-quality scores, and good reconstruction alone does not prove interpretability.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
axes[0].scatter(A[frame],R[frame],s=9,alpha=.5)
limits=[min(A[frame].min(),R[frame].min()),max(A[frame].max(),R[frame].max())]
axes[0].plot(limits,limits,'k--',lw=1);axes[0].set(xlabel='Original activation',ylabel='Reconstructed activation',title='768 coordinates at one frame')
error=np.linalg.norm(A-R,axis=1)/np.maximum(np.linalg.norm(A,axis=1),1e-12)
axes[1].plot(frame_times,error);axes[1].set(xlabel='Time (seconds)',ylabel='Relative L2 error',title='Reconstruction over time')
axes[2].plot(frame_times,(F>0).sum(axis=1));axes[2].axhline(int(sae.k.item()),ls='--',color='gray',label='Training target k')
axes[2].set(xlabel='Time (seconds)',ylabel='Active features',title='Sparsity over time');axes[2].legend()
fig.tight_layout();show(fig,'04_reconstruction_sparsity')


## 6. Which features respond across your recordings?

We compute the maximum, mean, and firing frequency of every feature for every selected recording. Max activation answers “did it respond strongly anywhere?”; firing frequency answers “how often was it nonzero?” These are different measurements.

A feature that never fires in this small subset is **unobserved here**, not necessarily globally dead. The cache stores derived statistics; changing selected files, model, threshold, or helper code changes its key.


In [ ]:
helper_hashes={str(p.relative_to(BASE)):sha(p) for p in sorted((BASE/'vendor').rglob('*.py'))}
fingerprint={'records':records,'audio_stat':[(r['wavID'],Path(r['path']).stat().st_size,Path(r['path']).stat().st_mtime_ns) for r in records], 'sr':SR,'max_seconds':MAX_SECONDS,'checkpoint':sha(CKPT),'helpers':helper_hashes}
cache_key=hashlib.sha256(json.dumps(fingerprint,sort_keys=True).encode()).hexdigest()[:16]
CACHE=BASE/f'activation_stats_{cache_key}.npz'
if CACHE.exists():
    cached=np.load(CACHE); max_acts=cached['max_acts'];mean_acts=cached['mean_acts'];freq=cached['freq'];clip_fve=cached['fve']
    print('Loaded matching cache',CACHE.name)
else:
    max_rows=[];mean_rows=[];freq_rows=[];fve_rows=[]
    for i,r in enumerate(records):
        wav=read_audio(r['path'])
        with torch.inference_mode():
            a=hubert(torch.from_numpy(wav)[None].to(DEVICE));f=sae.encode(a,use_threshold=True);rec=sae.decode(f)
        v=f[0].cpu().numpy()
        max_rows.append(v.max(axis=0));mean_rows.append(v.mean(axis=0));freq_rows.append((v>0).mean(axis=0))
        fve_rows.append(reconstruction_metrics(a,f,rec).frac_var_explained)
        if (i+1)%8==0:print(f'Processed {i+1}/{len(records)} recordings')
    max_acts=np.stack(max_rows);mean_acts=np.stack(mean_rows);freq=np.stack(freq_rows);clip_fve=np.array(fve_rows)
    np.savez_compressed(CACHE,max_acts=max_acts,mean_acts=mean_acts,freq=freq,fve=clip_fve)
print('Per-recording peak activation matrix:',max_acts.shape)
print('Features never activated on this subset:',int((max_acts.max(axis=0)==0).sum()))
print('Median per-clip variance explained:',round(float(np.median(clip_fve)),3))
feature_id=int(max_acts.max(axis=0).argmax())
order=np.argsort(-max_acts[:,feature_id])[:5]
print('Feature selected by largest corpus peak:',feature_id,'(an exploratory choice, not a selectivity test)')
for i in order:print(records[i]['emotion'],records[i]['wavID'],'peak=',round(float(max_acts[i,feature_id]),3))


## 7. Align one feature with the waveform, spectrogram, and sound

We now use the recording with the strongest peak for the chosen feature. The vertical line marks its highest response. Listen to the original excerpt around that point and look for an acoustic pattern. Compare several examples before suggesting a name.

Frames are placed on an approximate 50 Hz timeline (one step every 20 ms); encoder receptive fields span a wider context. Peaks therefore do not identify an exact causal onset.


In [ ]:
best_i=int(order[0]); best_record=records[best_i]; best_wave=read_audio(best_record['path'])
with torch.inference_mode():
    best_a=hubert(torch.from_numpy(best_wave)[None].to(DEVICE));best_f=sae.encode(best_a,use_threshold=True)[0].cpu().numpy()
times=np.arange(len(best_f))/50.0
peak_frame=int(best_f[:,feature_id].argmax());peak_time=peak_frame/50.0
best_mel=librosa.power_to_db(librosa.feature.melspectrogram(y=best_wave,sr=SR,n_mels=80,hop_length=160),ref=np.max)
fig,axes=plt.subplots(3,1,figsize=(12,6),sharex=True)
axes[0].plot(np.arange(len(best_wave))/SR,best_wave,lw=.5);axes[0].set_ylabel('Amplitude')
axes[0].set_title(f"F{feature_id} | {best_record['emotion']} | {best_record['wavID']}")
axes[1].imshow(best_mel,aspect='auto',origin='lower',extent=[0,len(best_wave)/SR,0,80],cmap='magma');axes[1].set_ylabel('Mel band')
axes[2].plot(times,best_f[:,feature_id],color='#c36b20');axes[2].set(xlabel='Time (seconds)',ylabel=f'F{feature_id} response')
for ax in axes:ax.axvline(peak_time,color='#c23c3c',ls='--',lw=1)
fig.tight_layout();show(fig,'05_feature_timeline')
for i in order[:3]:
    r=records[int(i)];w=read_audio(r['path'])
    with torch.inference_mode(): z=sae.encode(hubert(torch.from_numpy(w)[None].to(DEVICE)),use_threshold=True)[0]
    t=int(z[:,feature_id].argmax())/50
    lo=max(0,int((t-.5)*SR));hi=min(len(w),int((t+.5)*SR))
    print(f"{r['emotion']} | peak at {t:.2f}s | {r['wavID']}")
    display(Audio(w[lo:hi],rate=SR,normalize=False))
    display(Audio(w[:],rate=SR,normalize=False))


### Compare the spectrogram and F1465 for all three audio examples

Rows are **Angry, Neutral, Sad**; columns are the **log-mel spectrogram** and **F1465 activation**. Each dashed line marks that recording's activation peak. In the first figure the gray band marks the excerpt used by its audio player. The second figure zooms to those exact excerpt boundaries. Horizontal axes retain the original recording times; each excerpt surrounds its own peak, not the same spoken word.

All activation panels share the same vertical scale. Spectrogram colors use −80 to 0 dB relative to each clip's own peak; these are comparable patterns, not absolute loudness measurements. Feature activations are sampled at approximately 50 Hz. These recordings differ in speaker, system, and text, so shared activation does not establish an emotion-specific feature.


In [ ]:
# Compare F1465 in the same three audio examples (not three emotion prototypes).
import librosa
comparison_feature = 1465
comparison_names = [
    'vmc2026-track2-sys011-spk026-utt046.wav',
    'vmc2026-track2-sys003-spk010-utt016.wav',
    'vmc2026-track2-sys007-spk015-utt106.wav',
]
comparison_records = {r['wavID']:r for r in records}
comparison_data = []
for name in comparison_names:
    record = comparison_records[name]
    wave = read_audio(record['path'])
    with torch.inference_mode():
        hidden = hubert(torch.from_numpy(wave)[None].to(DEVICE))
        response = sae.encode(hidden, use_threshold=True)[0,:,comparison_feature].cpu().numpy()
    times = np.arange(len(response))/50.0
    peak_time = float(times[response.argmax()])
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=wave, sr=SR, n_mels=80, hop_length=160), ref=np.max)
    # Use the exact sample boundaries of the already embedded audio players.
    lo = max(0, int((peak_time-.5)*SR)); hi = min(len(wave), int((peak_time+.5)*SR))
    comparison_data.append(dict(record=record,wave=wave,times=times,response=response,peak=peak_time,
                                mel_db=mel_db,start=lo/SR,end=hi/SR))
shared_max_response = max(float(d['response'].max()) for d in comparison_data)*1.12
shared_max_duration = max(len(d['wave'])/SR for d in comparison_data)
for cropped, figure_name in [(False,'10_f1465_three_recordings'),(True,'11_f1465_three_audio_excerpts')]:
    fig, axes = plt.subplots(3,2,figsize=(14,9),gridspec_kw={'width_ratios':[1.5,1]}, squeeze=False)
    for row,d in enumerate(comparison_data):
        duration = len(d['wave'])/SR
        left, right = axes[row]
        left.imshow(d['mel_db'],origin='lower',aspect='auto',extent=[0,duration,0,80],
                    cmap='magma',vmin=-80,vmax=0,interpolation='nearest')
        right.plot(d['times'],d['response'],color='#c36b20',lw=1.8)
        bounds = (d['start'],d['end']) if cropped else (0,shared_max_duration)
        for ax in (left,right):
            ax.set_xlim(*bounds)
            ax.axvline(d['peak'],color='#b83232',ls='--',lw=1)
            ax.set_xlabel('Time in original recording (seconds)')
            if not cropped:ax.axvspan(d['start'],d['end'],color='#888888',alpha=.12)
        left.set_ylabel('Mel band')
        right.set_ylabel('F1465 activation');right.set_ylim(0,shared_max_response)
        label=d['record']['emotion']
        left.set_title(f"{label}: log-mel spectrogram | {d['record']['wavID'].split('-')[-2]}")
        right.set_title(f"{label}: peak {d['response'].max():.2f} at {d['peak']:.2f} s")
    fig.suptitle('F1465: exact audio-player excerpts' if cropped else 'F1465: three recordings, with audio-player excerpts shaded',y=.995)
    fig.tight_layout(rect=[0,0,1,.97])
    show(fig,figure_name)
for d in comparison_data:
    print(f"{d['record']['emotion']}: peak {d['response'].max():.3f} at {d['peak']:.2f}s; audio excerpt {d['start']:.2f}–{d['end']:.2f}s")


### Additional example: sys014 / spk006 / utt136

This cell checks the explicitly requested recording with the same pretrained HuBERT SAE and inference threshold. Read the metadata text, listen to the full recording and the peak excerpt, and compare the F1465 timing. The dashed line is an activation peak, **not an automatically aligned word boundary**. This checks a candidate example; a response alone does not establish that F1465 detects the word “came”.


In [ ]:
# Additional F1465 example: explicitly requested recording.
import csv
import librosa
additional_name = 'vmc2026-track2-sys014-spk006-utt136.wav'
additional_path = DATA / 'wav' / additional_name
assert additional_path.is_file(), additional_path
additional_metadata = None
with open(DATA/'metadata.csv', newline='') as metadata_stream:
    for row in csv.reader(metadata_stream, delimiter='|'):
        if row and row[0] == additional_name:
            additional_metadata = {'emotion': row[1], 'text': '|'.join(row[2:])}
            break
print('Recording:', additional_name)
print('Metadata:', additional_metadata)
additional_wave = read_audio(additional_path)
print(f'Analyzed duration: {len(additional_wave)/SR:.2f} seconds (existing duration cap applies)')
with torch.inference_mode():
    additional_hidden = hubert(torch.from_numpy(additional_wave)[None].to(DEVICE))
    additional_features = sae.encode(additional_hidden, use_threshold=True)[0].cpu().numpy()
additional_response = additional_features[:,1465]
additional_times = np.arange(len(additional_response))/50.0
additional_active = additional_response > 0
additional_peak = float(additional_response.max())
additional_peak_time = float(additional_times[additional_response.argmax()]) if additional_peak > 0 else None
print(f'F1465 maximum activation: {additional_peak:.4f}')
print(f'F1465 active frames: {additional_active.sum()}/{len(additional_active)} ({additional_active.mean():.2%})')
print('F1465 peak time:', f'{additional_peak_time:.2f} seconds' if additional_peak_time is not None else 'No nonzero response')
additional_mel = librosa.power_to_db(librosa.feature.melspectrogram(y=additional_wave, sr=SR, n_mels=80, hop_length=160), ref=np.max)
additional_duration = len(additional_wave)/SR
fig, axes = plt.subplots(2,1,figsize=(12,6),sharex=True)
im = axes[0].imshow(additional_mel,origin='lower',aspect='auto',extent=[0,additional_duration,0,80],cmap='magma',vmin=-80,vmax=0)
axes[0].set_ylabel('Mel band')
axes[0].set_title(f"{additional_name} | {(additional_metadata or {}).get('emotion','unknown emotion')}")
axes[1].plot(additional_times,additional_response,color='#c36b20',lw=1.8)
axes[1].set(xlabel='Time in recording (seconds)',ylabel='F1465 activation',xlim=(0,additional_duration),ylim=(0,max(6.3,additional_peak*1.12)))
if additional_peak_time is not None:
    for ax in axes:ax.axvline(additional_peak_time,color='#b83232',ls='--',lw=1)
fig.tight_layout()
show(fig,'12_f1465_sys014_spk006_utt136')
print('Full analyzed audio:')
display(Audio(additional_wave,rate=SR,normalize=False))
if additional_peak_time is not None:
    additional_lo=max(0,int((additional_peak_time-.5)*SR));additional_hi=min(len(additional_wave),int((additional_peak_time+.5)*SR))
    print(f'Peak excerpt: {additional_lo/SR:.2f}–{additional_hi/SR:.2f} seconds')
    display(Audio(additional_wave[additional_lo:additional_hi],rate=SR,normalize=False))
print('A response here is evidence to investigate; identifying the word came requires checking its timing and testing other words/voices.')


### Five additional came examples: F1465 and approximate word timing

Five new recordings were selected **before looking at their activations**, one per intended emotion and with preference for distinct systems/speakers. The original sys014/spk006/utt136 example is a separate reference. The selection is saved in `came_f1465_check/selection.json` and is reused on reruns.

All 467 available came examples use the same sentence, **“Rat came and replied on the leaves.”** These recordings test repeatability across voices and emotions, but cannot isolate the word from its phonemes, neighboring words, or sentence context.

The independent English ASR model `facebook/wav2vec2-base-960h` supplies character posteriors for CTC alignment against the complete metadata transcript. Green shading marks the **estimated** came interval; red dashed lines mark F1465 global peaks. The greedy ASR output is also displayed as a check. Word times are approximate, and forced alignment assumes the supplied transcript; it does not independently prove what was spoken. Listen to the full audio, estimated word excerpt, and feature-peak excerpt before interpreting overlap.

This is the pretrained **HuBERT SAE** from the learning notebook, with the same checkpoint and extraction location. It is not the later whole-recording M2D-CLAP SAE. A feature's lack of response in a came recording is retained, not discarded. No feature meaning is automatically confirmed.


In [ ]:
# FIVE_NEW_CAME_EXAMPLES_F1465
from pathlib import Path
import sys
SAE_BASE = Path(os.environ.get("SPEECHSAE_BASE", str(Path.cwd())))
if str(SAE_BASE) not in sys.path:
    sys.path.insert(0, str(SAE_BASE))
from came_feature_check import run_came_checks
came_f1465_summary = run_came_checks(SAE_BASE)


In [ ]:
# Original recording: 0.80–1.30 seconds.
# The activation peak is 0.26 seconds into this excerpt.
display(Audio(
    additional_wave[int(0.80 * SR):int(1.30 * SR)],
    rate=SR,
    normalize=False
))

In [ ]:
# For a tensor shaped [recording, time frame, feature]
f1465_at_one_frame = feats[0, frame, 1465]

# F1465's responses across the whole recording
f1465_over_time = feats[0, :, 1465]

## 8. A spectrogram pattern around feature responses

Following the original notebook, extract windows centered on active frames and average activation-weighted log-mel windows. Here `window_s=0.5` means **0.5 seconds on either side** (approximately 1 second total). The middle is the response time.

The plot is an average of surrounding audio structure, not a feature-activation trace or causal receptive field. Overlapping windows are correlated; multiplying negative dB values by activations also changes their scale. Use this as a visual clue together with the actual audio, not a quantitative energy comparison.


In [ ]:
windows=feature_activation_windows(torch.from_numpy(best_wave)[None],torch.from_numpy(best_f),feature_idx=feature_id,sample_rate=SR,frame_rate_hz=50.0,window_s=.5)
if windows:
    avg=np.stack(windows).mean(axis=0)
    fig,ax=plt.subplots(figsize=(8,4))
    im=ax.imshow(avg,aspect='auto',origin='lower',extent=[-.5,.5,0,80],cmap='magma')
    ax.axvline(0,color='cyan',ls='--',lw=1);ax.set(xlabel='Time relative to active frame (seconds)',ylabel='Mel band',title=f'F{feature_id}: average of {len(windows)} weighted windows')
    fig.colorbar(im,ax=ax,label='Activation-weighted log-mel value');fig.tight_layout();show(fig,'06_average_activation_window')
else:print('No valid centered windows; try another feature or a longer recording.')


## 9. Explore associations with intended emotion

Your subset consists of emotional TTS recordings, so it would be misleading to copy the original speech/music/sounds labeling. Instead we compare mean feature firing frequency across **intended emotion groups**, averaging recordings equally.

This is descriptive: speaker, system, text, duration, and recording conditions may explain a group difference. Emotion association does not establish an emotion detector or a causal emotion mechanism. No significance or generalization claim follows from this small exploratory subset.


In [ ]:
emotions=sorted(set(r['emotion'] for r in records));emotion_arr=np.array([r['emotion'] for r in records])
emo_freq=np.stack([freq[emotion_arr==e].mean(axis=0) for e in emotions])
active_ids=np.flatnonzero(max_acts.max(axis=0)>0)
variability=emo_freq[:,active_ids].std(axis=0)
view_ids=active_ids[np.argsort(variability)[-16:][::-1]]
fig,ax=plt.subplots(figsize=(12,4))
im=ax.imshow(emo_freq[:,view_ids],aspect='auto',cmap='viridis',vmin=0)
ax.set_xticks(range(len(view_ids)),[f'F{i}' for i in view_ids],rotation=55,ha='right');ax.set_yticks(range(len(emotions)),emotions)
ax.set_title('Features with largest exploratory variation in emotion-group firing frequency')
fig.colorbar(im,ax=ax,label='Mean fraction of frames active');fig.tight_layout();show(fig,'07_emotion_feature_frequencies')


## 10. Map feature detectors in two dimensions (t-SNE)

Each point below is **one SAE feature's encoder weight vector**, not a recording, emotion, or time frame. We show up to 512 frequently observed features for a readable, quick map. A color is assigned only when the leading emotion frequency exceeds the next by at least 0.03; otherwise it is “mixed”. This threshold is illustrative, not a statistical test.

t-SNE emphasizes some local neighborhoods and distorts other geometry. Its axes have no acoustic units. A cluster is a starting point for inspection, not proof of a semantic category.


In [ ]:
from sklearn.manifold import TSNE
map_ids=active_ids[np.argsort(freq[:,active_ids].mean(axis=0))[-min(512,len(active_ids)):]]
weights=sae.encoder.weight.detach().cpu().numpy()[map_ids]
coords=TSNE(n_components=2,perplexity=min(25,(len(map_ids)-1)/3),init='pca',learning_rate='auto',random_state=SEED).fit_transform(weights)
sorted_freq=np.sort(emo_freq[:,map_ids],axis=0)
leads=sorted_freq[-1]-sorted_freq[-2]
labels=np.array([emotions[i] for i in emo_freq[:,map_ids].argmax(axis=0)],dtype=object);labels[leads<.03]='mixed'
fig,ax=plt.subplots(figsize=(9,6))
for i,label in enumerate(['mixed']+emotions):
    mask=labels==label
    if mask.any():ax.scatter(coords[mask,0],coords[mask,1],s=15,alpha=.55,c='gray' if label=='mixed' else f'C{i-1}',label=f'{label} ({mask.sum()})')
ax.set(xlabel='t-SNE coordinate 1',ylabel='t-SNE coordinate 2',title=f'{len(map_ids)} SAE feature detectors: exploratory emotion association')
ax.legend(bbox_to_anchor=(1.02,1),loc='upper left');fig.tight_layout();show(fig,'08_feature_tsne')


## 11. What you can conclude, and how this connects to TransportMOS

You have now observed real activations on your own recordings, checked how much information the sparse reconstruction preserves, identified audio that activates a feature, and explored associations across emotions.

You have **not yet** established what a feature means, that changing it changes perception, or that it explains your MOS evaluator. To investigate these questions, collect many diverse activating and nonactivating examples, control for speaker/system/text, and validate proposed interpretations on recordings not used to discover them.

For TransportMOS specifically, train or obtain an SAE on the **same encoder and exact layer used by your actual evaluator**. Then test feature interventions and their effects on evaluator predictions, and compare with human judgments. Simply attaching this HuBERT SAE to a WavLM/M2D2 representation would be invalid.

**Try next:** change `feature_id` to one of the IDs in the emotion-frequency plot and rerun sections 7–8. Listen before naming it. Re-run section 2 with a different fixed seed to see how sensitive your interpretation is to the recording selection.


## Reading your saved run

- **Data:** 40 real training recordings: 8 each of Angry, Happy, Neutral, Sad, and Surprise.
- **Example:** the Happy recording `vmc2026-track2-sys005-spk002-utt097.wav` is 2.34 seconds long. HuBERT produces 116 time frames, each with 768 coordinates. The SAE represents each frame with 6,144 possible features.
- **Dense-to-sparse plot:** at about 0.90 seconds, 45 of those 6,144 features are nonzero. The bar plot magnifies the ten strongest responses; these are magnitudes, not probabilities.
- **Reconstruction:** the Happy clip averages 46.3 active features per frame and 85.7% fraction of activation variance explained. The median per-clip value over the 40 clips is 87.7%. These are hidden-state fidelity measures, not percentages of speech quality or model understanding.
- **Feature F1465:** this feature was chosen because it had the largest observed peak, not because it was established as an emotion detector. In the strongest example, an Angry recording, it responds around 0.50 seconds. It also fires in a Neutral and a Sad recording; this already shows why calling it an "Angry feature" would be premature. The last two zero-valued entries in the five-clip ranking are non-firing examples, not additional positive matches.
- **Not seen here:** 2,015 features never activate in this small subset. Other recordings may activate them.
- **Your next exercise:** compare the three embedded peak excerpts for F1465. Write down what they share acoustically; then look for counterexamples before assigning a feature name.


In [ ]:
manifest={'purpose':'Educational exploratory analysis, not a confirmatory evaluation','seed':SEED,'selected_recordings':len(records),'seconds_per_clip_cap':MAX_SECONDS,'sample_rate':SR,'python':sys.executable,'torch':torch.__version__,'device':DEVICE,'checkpoint':str(CKPT),'checkpoint_sha256':sha(CKPT),'model':'facebook/hubert-base-ls960','checkpoint_layer_label':LAYER,'hf_hidden_state_index':hubert.sae_after_layer,'feature_count':sae.dict_size,'sparsity_target_k':int(sae.k.item()),'inference_threshold':float(sae.threshold.item()),'example_metrics':vars(metrics),'selected_feature':feature_id,'cache':CACHE.name,'source_file_sha256':helper_hashes,'figures':[p.name for p in sorted(FIGS.glob('*.png'))]}
(BASE/'walkthrough_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved walkthrough_manifest.json and',len(manifest['figures']),'figures')


In [ ]:
# After during the investigation with other recording like vmc2026-track2-sys014-spk006-utt136.wav


# F1465 peak, I hear "came".

## Your own recording: me_1.wav and F1465

F1465 activated during the approximately aligned word **came** in your recording.

Independent greedy ASR transcribed:

> How do i know you came to my house

| Recording | Estimated came (s) | F1465 peak (s) | Peak activation | Peak overlaps came? | ASR contains came? |
| :--- | ---: | ---: | ---: | :--- | :--- |
| me_1.wav | 2.52–2.74 | 2.58 | 4.721 | Yes | Yes |

This adds evidence that the association extends to a different sentence and your voice. Tests on recordings without “came” are still needed to establish specificity. Word timings are approximate and use the ASR transcript, not a human-verified transcript.

Green marks estimated “came”; the red dashed line marks F1465’s peak.

Results: `audio_recordbyme/me_1_analysis/`. The input has M4A content despite its `.wav` extension; a separate decoded copy was used for analysis.


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
# Play the decoded copy of your original recording.
me_audio = BASE / 'audio_recordbyme/me_1_analysis/decoded_16k.wav'
display(Audio(filename=str(me_audio)))


In [ ]:
from pathlib import Path
from IPython.display import display, Image
me_analysis = BASE / 'audio_recordbyme/me_1_analysis'
display(Image(filename=str(me_analysis / 'me_1_f1465.png')))
